In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when
from pyspark.ml import PipelineModel


def init_spark_session(app_name: str = "CommercialLoanRiskInference") -> SparkSession:
    """Initializes a Spark session for local batch scoring."""
    SparkSession.builder.getOrCreate().stop()
    
    return SparkSession.builder \
        .appName(app_name) \
        .config("spark.driver.host", "localhost") \
        .config("spark.sql.shuffle.partitions", "4") \
        .getOrCreate()


def preprocess_test_data(df):
    """Preprocesses raw test dataset to match training schema features."""
    fill_defaults = {
        "Gender": "Male",
        "Married": "Yes",
        "Dependents": "0",
        "Self_Employed": "No",
        "LoanAmount": 146.4,
        "Loan_Amount_Term": 360,
        "Credit_History": 1.0
    }
    
    return df.na.fill(fill_defaults) \
        .withColumn("Total_Income", col("ApplicantIncome") + col("CoapplicantIncome")) \
        .withColumn(
            "Debt_To_Income",
            when(col("Total_Income") > 0, col("LoanAmount") / col("Total_Income")).otherwise(0.0)
        )


def main():
    spark = init_spark_session()

    model_path = r"C:\Users\Lincoln\Downloads\Data\production_loan_model"
    test_data_path = r"C:\Users\Lincoln\Downloads\Data\2_Test_Data.csv"
    output_save_path = r"C:\Users\Lincoln\Downloads\Data\loan_predictions_csv"

    # Load Artifacts and Ingest Data
    pipeline_model = PipelineModel.load(model_path)
    raw_test_df = spark.read.csv(test_data_path, header=True, inferSchema=True)

    # Preprocessing and Batch Inference
    test_cleaned = preprocess_test_data(raw_test_df)
    predictions_df = pipeline_model.transform(test_cleaned)

    # Append Business Decision Flag
    annotated_df = predictions_df.withColumn(
        "Decision_Flag",
        when(col("prediction") == 1.0, "APPROVED").otherwise("REJECTED")
    )

    # Select Key Columns for Clean CSV Export
    export_df = annotated_df.select(
        "Loan_ID",
        "ApplicantIncome",
        "CoapplicantIncome",
        "LoanAmount",
        "Credit_History",
        "Debt_To_Income",
        "prediction",
        "Decision_Flag"
    )

    # Preview Output
    export_df.show(10, truncate=False)

    # Persist Results as CSV (coalesced into a single CSV file)
    export_df.coalesce(1).write.mode("overwrite").option("header", "true").csv(output_save_path)
    print(f"Batch scoring completed successfully. CSV saved to {output_save_path}")


if __name__ == "__main__":
    main()

+--------+---------------+-----------------+----------+--------------+--------------------+----------+-------------+
|Loan_ID |ApplicantIncome|CoapplicantIncome|LoanAmount|Credit_History|Debt_To_Income      |prediction|Decision_Flag|
+--------+---------------+-----------------+----------+--------------+--------------------+----------+-------------+
|LP001015|5720           |0                |110       |1             |0.019230769230769232|1.0       |APPROVED     |
|LP001022|3076           |1500             |126       |1             |0.027534965034965036|1.0       |APPROVED     |
|LP001031|5000           |1800             |208       |1             |0.03058823529411765 |1.0       |APPROVED     |
|LP001035|2340           |2546             |100       |1             |0.020466639377814164|1.0       |APPROVED     |
|LP001051|3276           |0                |78        |1             |0.023809523809523808|1.0       |APPROVED     |
|LP001054|2165           |3422             |152       |1        